# 8 — Multi-Agent Research Pipeline
> Turn any research topic into a polished executive brief — automatically.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


# --- Data shapes ---

class ResearchFindings(BaseModel):
    topic: str
    key_facts: List[str] = Field(description="3-5 concrete facts with data or citations where possible")
    trends: List[str] = Field(description="2-3 observed trends or emerging developments")
    gaps: List[str] = Field(description="Questions this research could not answer — limits and unknowns")


class WrittenBrief(BaseModel):
    title: str
    executive_summary: str = Field(description="2-3 sentence summary for a busy executive")
    body: str = Field(description="400-600 word brief in markdown format")
    key_takeaways: List[str] = Field(description="3 bullet points the reader should remember")
    further_reading: List[str] = Field(description="2-3 topic areas worth exploring next")


# --- Agent prompts ---

SUPERVISOR_SYSTEM = SystemMessage(
    """You are a research supervisor. You receive a topic and decide how to frame
the research question for the researcher sub-agent.

Turn vague topics into focused, answerable research questions.
Example: "AI in healthcare" -> "What are the current applications and adoption barriers of AI in clinical diagnosis?"
Return only the refined research question — one sentence."""
)

RESEARCHER_SYSTEM = SystemMessage(
    """You are a research analyst. Given a topic, gather what you know and produce structured findings.

Be factual and specific. Prefer concrete data points over vague generalizations.
If you are uncertain about a fact, flag it in gaps rather than stating it as fact.
Do not invent statistics."""
)

WRITER_SYSTEM = SystemMessage(
    """You are a business writer who turns research findings into clear, executive-level briefs.

Style:
- Lead with the most important insight
- Use plain English — no jargon
- Structure: executive summary -> context -> key findings -> implications -> takeaways
- Markdown formatting with ## headers
- Do not repeat verbatim content from the findings; synthesize and add framing"""
)


# --- Pipeline ---

def run(topic: str) -> dict:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Stage 1: Supervisor refines the question
    supervisor = SUPERVISOR_SYSTEM | llm
    refined = supervisor.invoke(HumanMessage(content=f"Topic: {topic}"))
    refined_question = refined.content.strip()

    # Stage 2: Researcher gathers findings
    researcher = RESEARCHER_SYSTEM | llm.with_structured_output(ResearchFindings)
    research: ResearchFindings = researcher.invoke(
        f"Research question: {refined_question}"
    )

    # Stage 3: Writer turns findings into a brief
    writer_input = (
        f"Research question: {refined_question}\n\n"
        f"Key facts:\n" + "\n".join(f"- {f}" for f in research.key_facts) + "\n\n"
        "Trends:\n" + "\n".join(f"- {t}" for t in research.trends) + "\n\n"
        "Gaps:\n" + "\n".join(f"- {g}" for g in research.gaps)
    )
    writer = WRITER_SYSTEM | llm.with_structured_output(WrittenBrief)
    brief: WrittenBrief = writer.invoke(writer_input)

    return {
        "refined_question": refined_question,
        "research": research,
        "brief": brief,
    }

## The scenario

A strategy team needs a quick brief on whether battery storage is ready to support commercial real estate going off-grid — without spending two days reading industry reports. We drop in the topic, and three agents handle the rest: one sharpens the question, one pulls the facts and flags what is still unknown, and one writes the brief ready for leadership.

In [ ]:
TOPIC = "Battery storage viability for commercial real estate energy independence"

print(f"Topic: {TOPIC}")
print("Running: Supervisor -> Researcher -> Writer...")
print("-" * 60)

result = run(TOPIC)

print("\n[SUPERVISOR] Refined question:")
print(f"  {result['refined_question']}")

research = result["research"]
print(f"\n[RESEARCHER] Findings on: {research.topic}")
print("\n  Key facts:")
for f in research.key_facts:
    print(f"    - {f}")
print("\n  Trends:")
for t in research.trends:
    print(f"    - {t}")
print("\n  Gaps:")
for g in research.gaps:
    print(f"    - {g}")

brief = result["brief"]
print(f"\n{'=' * 60}")
print(f"[WRITER] {brief.title}")
print(f"{'=' * 60}")
print(f"\nExecutive Summary:\n  {brief.executive_summary}")
print("\nKey Takeaways:")
for kt in brief.key_takeaways:
    print(f"  * {kt}")
print("\nFurther Reading:")
for fr in brief.further_reading:
    print(f"  - {fr}")
print("\n--- Full Brief ---\n")
print(brief.body)

## Use your own data

Replace the input above with:
- your research topic (one sentence or phrase is enough)
- any industry, technology, market, or strategic question your team is tracking

The pipeline will return a sharpened research question, sourced facts and trends, identified gaps, and a full executive brief ready to share.